In [1]:
# before running this notebook, make sure to do the following
# (1) activate optbinning virtual environment (run this on terminal: .venv-optbinning\Scripts\Activate.ps1)
# (2) select optbinning kernel
# (3) 1.7.2 version of Sklearn is required for optbinning 0.20.0 (latest) to work.

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import pickle
import sklearn
import optbinning
import pandas as pd
import matplotlib.pyplot as plt
from optbinning import OptimalBinning

In [ ]:
print(sklearn.__version__)
print(optbinning.__version__)

In [ ]:
train_data = pd.read_parquet(
    "../data/processed/train_eda_output.parquet"
)

In [ ]:
train_data.head()

In [ ]:
# Useful variables: 'n_dpd_90plus_hist', 'n_dpd_60_89_l2yrs', 'n_dpd_30_50_l2yrs', 'avg_utli_unsec', 'age'
# Weak variables: 'n_dependents_imp', 'monthly_income_imp'
# Useless variables: 'n_mort_loans', 'n_credit_lines', 'debt_income_ratio', 'monthly_income_nan_flag'

In [ ]:
def compute_iv(var_name):
    if var_name in ("avg_util_unsec", "age_imp", "monthly_income_imp", "debt_income_ratio"):
        dtype = "numerical"
        var_type = "continuous"
    else:
        dtype = "numerical"
        var_type = "ordinal"

    binning = OptimalBinning(name=var_name, dtype=dtype)
    binning.fit(train_data[var_name], train_data["f_dpd_90plus_nxt_2yrs"])
    binning_table = binning.binning_table.build()
    iv = binning_table["IV"].iloc[-1]
    return iv, var_type

In [ ]:
variables = [
    'n_dpd_90plus_hist_imp', 'n_dpd_60_89_l2yrs_imp', 'n_dpd_30_50_l2yrs_imp',
    'avg_util_unsec', 'age_imp', 'n_dependents_imp', 'monthly_income_imp',
    'n_mort_loans', 'n_credit_lines', 'debt_income_ratio', 'monthly_income_nan_flag'
]

iv_results = []
for var in variables:
    iv, var_type = compute_iv(var)
    iv_results.append((var, var_type, iv))

iv_df = pd.DataFrame(iv_results, columns=["var_name", "var_type", "iv"])
iv_df.sort_values(by="iv", ascending=False, inplace=True)
iv_df

In [ ]:
def calc_woe_trend(var_name):
    if var_name in ("avg_util_unsec", "age_imp", "monthly_income_imp", "debt_income_ratio"):
        dtype = "numerical"
        var_type = "continuous"
    else:
        dtype = "numerical"
        var_type = "ordinal"

    binning = OptimalBinning(
        name=var_name,
        dtype=dtype
    )

    binning.fit(train_data[var_name], train_data["f_dpd_90plus_nxt_2yrs"])
    binning_table = binning.binning_table.build()
    binning_table.insert(0, "variable", var_name)
    return binning_table[["variable", "Bin", "WoE"]]

In [ ]:
def plot_woe_trend(input_df):
    plot_df =  input_df.iloc[:-1]
    plot_title =  plot_df.at[0, "variable"]

    plt.figure(figsize=(10, 2))
    plt.plot(plot_df["Bin"], plot_df["WoE"], marker="o")
    plt.title(plot_title)
    plt.xlabel("Bin")
    plt.ylabel("WoE")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()

In [ ]:
#checking WOE trend for impn vars
plot_woe_trend(
    input_df = calc_woe_trend("n_dpd_90plus_hist_imp")
)
plot_woe_trend(
    input_df = calc_woe_trend("n_dpd_30_50_l2yrs_imp")
)
plot_woe_trend(
    input_df = calc_woe_trend("avg_util_unsec")
)
plot_woe_trend(
    input_df = calc_woe_trend("age_imp")
)
plot_woe_trend(
    input_df = calc_woe_trend("monthly_income_imp")
)
plot_woe_trend(
    input_df = calc_woe_trend("debt_income_ratio")
)

In [ ]:
impn_vars_logit = [
    "n_dpd_90plus_hist_imp",
    "n_dpd_30_50_l2yrs_imp",
    "avg_util_unsec",
    "age_imp",
    "monthly_income_imp",
    "debt_income_ratio"
]

iv_df.loc[iv_df["var_name"].isin(impn_vars_logit), :]

In [ ]:
train_logit_data = train_data.loc[:, ["acct_id"] + impn_vars_logit + ["f_dpd_90plus_nxt_2yrs"]].copy(deep=True)

In [ ]:
train_logit_data.head(3)

In [ ]:
def woe_transformation(
    var_list,
    target="f_dpd_90plus_nxt_2yrs",
    train_data=train_logit_data
):

    woe_transformers = {}

    for var_name in var_list:

        binning = OptimalBinning(
            name=var_name,
            dtype="numerical"
        )

        binning.fit(
            train_data[var_name],
            train_data[target]
        )

        # Save fitted binning object
        woe_transformers[var_name] = binning

        # Transform training data
        woe_trans_series = binning.transform(
            train_data[var_name],
            metric="woe"
        )

        train_data[f"{var_name}_woe"] = woe_trans_series

    return train_data, woe_transformers

In [ ]:
train_raw_woetrns_data, woe_transformers = woe_transformation(
    var_list=impn_vars_logit
)

train_logit_woetrns_data = train_raw_woetrns_data.drop(
    columns=impn_vars_logit
)

train_logit_woetrns_data.head(3)

In [ ]:
display(woe_transformers)
display(woe_transformers["n_dpd_90plus_hist_imp"].binning_table.build())

In [ ]:
file_path = "../data/processed/train_woe_output.parquet"

if os.path.exists(file_path):
    print("File already exists!")
else:
    train_logit_woetrns_data.to_parquet(
        file_path,
        index=False
    )
    print("File created successfully!")

In [ ]:
woe_transformer_path = "../models/woe_transformer.pkl"

if os.path.exists(woe_transformer_path):
    print("WOE transformers already exist!")
else:
    with open(woe_transformer_path, "wb") as f:
        pickle.dump(woe_transformers, f)

    print("File saved successfully!")

In [ ]:
# train_data.head(3)

In [ ]:
# testing_df = train_data.loc[:, ["avg_util_unsec", "n_dpd_90plus_hist_imp", "n_dpd_30_50_l2yrs_imp", "age_imp", "monthly_income_imp", "debt_income_ratio"]].copy()
# testing_df.head(3)

In [ ]:
# avg_util_unsec
# n_dpd_90plus_hist_imp
# n_dpd_30_50_l2yrs_imp	
# age_imp
# debt_income_ratio

# woe_transformers

# testing_df = train_data.loc[0, ["avg_util_unsec", "n_dpd_90plus_hist_imp", "n_dpd_30_50_l2yrs_imp", "age_imp", "debt_income_ratio"]].copy()
# testing_df.head()

# display(woe_transformers["avg_util_unsec"].binning_table.build())
# display(woe_transformers["age_imp"].binning_table.build())

# testing_df["avg_util_unsec_woe"] = woe_transformers["avg_util_unsec"].transform(
#     testing_df["avg_util_unsec"],
#     metric="woe"
# )

# testing_df["age_imp_woe"] = woe_transformers["age_imp"].transform(
#     testing_df["age_imp"],
#     metric="woe"
# )

# testing_df